# Cross-trait and multi-polytranscriptional risk score analysis of PD in AMP-PD: Calculate PTS and run univariate analyses

**Project**: Cross-trait and multi-polytranscriptomic score analysis of Parkinson's disease identifies novel associations and improves prediction

**Date last updated**: July 2026 

 ## Initial set-up 

### Loading Python libraries

In [ ]:
# Use the os package to interact with the environment
import os
import sys

# Bring in Pandas for Dataframe functionality
import pandas as pd
from functools import reduce

# Bring some visualization functionality 
import seaborn as sns  

# numpy for basics
import numpy as np

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
%matplotlib inline

# Enable interaction with the FireCloud API
from firecloud import api as fapi

# Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.core.display import display, HTML

# Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

# BigQuery for querying data
from google.cloud import bigquery

#Import Sys
import sys as sys

### Defining functions

In [ ]:
# Utility routine for printing a shell command before executing it
def shell_do(command):
    print(f'Executing: {command}', file=sys.stderr)
    !$command
    
def shell_return(command):
    print(f'Executing: {command}', file=sys.stderr)
    output = !$command
    return '\n'.join(output)

# Utility routine for printing a query before executing it
def bq_query(query):
    print(f'Executing: {query}', file=sys.stderr)
    return pd.read_gbq(query, project_id=BILLING_PROJECT_ID, dialect='standard')

# Utility routine for display a message and a link
def display_html_link(description, link_text, url):
    html = f'''
    <p>
    </p>
    <p>
    {description}
    <a target=_blank href="{url}">{link_text}</a>.
    </p>
    '''

    display(HTML(html))

# Utility routines for reading files from Google Cloud Storage
def gcs_read_file(path):
    """Return the contents of a file in GCS"""
    contents = !gsutil -u {BILLING_PROJECT_ID} cat {path}
    return '\n'.join(contents)
    
def gcs_read_csv(path, sep=None):
    """Return a DataFrame from the contents of a delimited file in GCS"""
    return pd.read_csv(StringIO(gcs_read_file(path)), sep=sep, engine='python')

# Utility routine for displaying a message and link to Cloud Console
def link_to_cloud_console_gcs(description, link_text, gcs_path):
    url = '{}?{}'.format(
        os.path.join('https://console.cloud.google.com/storage/browser',
                     gcs_path.replace("gs://","")),
        urllib.parse.urlencode({'userProject': BILLING_PROJECT_ID}))

    display_html_link(description, link_text, url)

### Set paths

In [ ]:
# Set up billing project and data path variables
BILLING_PROJECT_ID = os.environ['GOOGLE_PROJECT']
WORKSPACE_NAMESPACE = os.environ['WORKSPACE_NAMESPACE']
WORKSPACE_NAME = os.environ['WORKSPACE_NAME']
WORKSPACE_BUCKET = os.environ['WORKSPACE_BUCKET']
WORKSPACE_ATTRIBUTES = fapi.get_workspace(WORKSPACE_NAMESPACE, WORKSPACE_NAME).json().get('workspace',{}).get('attributes',{})

## Print the information to check we are in the proper release and billing 
## This will be different for you, the user, depending on the billing project your workspace is on
print('Billing and Workspace')
print(f'Workspace Name @ `WORKSPACE_NAME`: {WORKSPACE_NAME}')
print(f'Billing Project @ `BILLING_PROJECT_ID`: {BILLING_PROJECT_ID}')
print(f'Workspace Bucket, where you can upload and download data @ `WORKSPACE_BUCKET`: {WORKSPACE_BUCKET}')
print('')

## AMP-PD v4.0
# Explicitly define release v4.0 path 
AMP_RELEASE_CASE_CONTROL_PATH = 'gs://amp-pd-data/releases/2023_v4release_1027'
AMP_PPMI_PDBP_TRANSCRIPTOMICS_RELEASE_PATH = 'gs://path/removed'
AMP_HBS_TRANSCRIPTOMICS_RELEASE_PATH = 'gs://path/removed'

#rnaseq_WB-RWTS-VHBS_samples.csv
#rnaseq_WB-RWTS_samples.csv


print('AMP-PD v4.0')
print(f'Path to AMP-PD v4.0 case/control data: {AMP_RELEASE_CASE_CONTROL_PATH}')
print(f'Path to AMP-PD v4.0 PPMI and PDBP RNA Data: {AMP_PPMI_PDBP_TRANSCRIPTOMICS_RELEASE_PATH}')
print(f'Path to AMP-PD v4.0 HBS Data: {AMP_HBS_TRANSCRIPTOMICS_RELEASE_PATH}')

### Make some directories and load R

In [ ]:
!ls /home/jupyter/multiTRS/

# Make a directories for the scorefiles
!mkdir -p /home/jupyter/multiTRS/scorefiles/
!ls /home/jupyter/multiTRS/
!mkdir -p /home/jupyter/multiTRS/scorefiles/TWAS_SMR_FDR_scorefiles
!ls /home/jupyter/multiTRS/scorefiles/

### Check the RNA data

In [ ]:
!ls /home/jupyter/multiTRS/RNA/clean/

In [ ]:
!pip install rpy2

In [ ]:
%load_ext rpy2.ipython

### Copy over PTS score files 

### Check files exists

In [ ]:
#Check the score files are in the main AMP-PD release 4 release path
shell_do(f'gsutil -u {BILLING_PROJECT_ID} ls {WORKSPACE_BUCKET}/uploads/TWAS_SMR_FDR_scores/')

### Copy over the score files

In [ ]:
# Copy over the case/control and demographics data etc to the working directory
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {WORKSPACE_BUCKET}/uploads/TWAS_SMR_FDR_scores/*.txt /home/jupyter/multiTRS/scorefiles/TWAS_SMR_FDR_scorefiles/')

### Check the files copied over

In [ ]:
#Check the data is in the directory
!ls /home/jupyter/multiTRS/scorefiles/TWAS_SMR_FDR_scorefiles/

## Calculate the PTS scores

### Make a directory for the scores

In [ ]:
!mkdir -p /home/jupyter/multiTRS/scores/
!ls /home/jupyter/multiTRS/

### Calculate the scores in each cohort

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)

# Set working directory, containing PTS scorefiles
setwd("/home/jupyter/multiTRS/scorefiles/TWAS_SMR_FDR_scorefiles/")

# List weights
weights_files <- list.files(pattern = ".txt")

# TEST
#weights_files <- weights_files[1:4]
#print(weights_files)

# Define the RNA path
rna_dir <- "/home/jupyter/multiTRS/RNA/clean"

# List RNA-seq files
rna_files <- list.files(path = rna_dir, pattern = "_baseline_cleaned_transposed_scaled.txt", full.names = TRUE
# TEST:
#rna_files <- rna_files[1]
#print(rna_files)

# Loop oover RNA-seq files to calculate the scores                       
for (i in rna_files){
    rna <- fread(i)
    cohort <- str_remove_all(i, "_baseline_cleaned_transposed_scaled.txt")
    cohort <- str_remove_all(cohort, "rnaseq_")
    cohort <- str_remove_all(cohort, "/home/jupyter/multiTRS/RNA/clean/")
    
    colnames(rna)[-1] <- str_replace(colnames(rna)[-1], "\\..*", "")
    
    rna_scores_final <- data.frame(participant_id = rna$participant_id)
    
    # We also want to know if any weight files are skipped and why
    skipped_weights <- data.table()
    # And we want to count how many features (genes) there are in each score
    n_features <- data.table()
    
    # Inner loop ver the PTS weights files
    for (j in weights_files){
        
        weights <- fread(j)
        
        pheno <- str_remove_all(j,"_scorefile.txt")
        
        # print(paste0("The phenotype being scored is ",pheno, " in ",cohort))
        
      if (nrow(weights) == 0) {
            skipped_weights <- rbind(
                skipped_weights, 
                data.table(weights_file = pheno, reason = "NA_sig_TWAS_SMR"))
          n_features <- rbind(
                n_features, 
                data.table(cohort = cohort, weights_file = pheno, n_weights = 0, n_features = 0))
            next
        }
        
        common_genes <- intersect(colnames(rna)[-1], weights$GENE)
        
        if(length(common_genes) == 0) {
        skipped_weights <- rbind(skipped_weights, data.table(weights_file = pheno, reason = "NA_common"))
        n_features <- rbind(n_features, data.table(cohort = cohort, weights_file = pheno, n_weights = nrow(weights), n_features = length(common_genes)))
        next
        }
        
        n_features <- rbind(n_features, data.table(cohort = cohort, weights_file = pheno, n_weights = nrow(weights), n_features = length(common_genes)))
        
        rna_genes_matched <- rna[, ..common_genes]
        
        weights_formatted <- setNames(weights$Z, weights$GENE)[common_genes]
        
        rna_genes_matched_scaled <- sweep(rna_genes_matched, 2, weights_formatted, `*`)
        
        summed_score <- rowSums(rna_genes_matched_scaled)

        rna_scores_final[[pheno]] <- summed_score
        
        
    }
    
    #print(head(rna_scores_final))
    print(skipped_weights)
    n_features <- n_features %>% arrange(desc(n_features))
    print(n_features)
    
    # Save the PTS scores, the list of skipped files and the features table
    write.table(rna_scores_final,paste0("/home/jupyter/multiTRS/scores/",cohort,"_TWAS_SMR_FDR_scores.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)
    write.table(skipped_weights,paste0("/home/jupyter/multiTRS/scores/",cohort,"_TWAS_SMR_FDR_skipped_list.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)
    write.table(n_features,paste0("/home/jupyter/multiTRS/scores/",cohort,"_TWAS_SMR_FDR_PTS_feature_list.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)
    
}

#### Check that the score files have been created

In [ ]:
!ls /home/jupyter/multiTRS/scores/

#### Copy the scores to the workspace directory

In [ ]:
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/scores/*_TWAS_SMR_FDR_scores.txt {WORKSPACE_BUCKET}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/scores/*_TWAS_SMR_FDR_skipped_list.txt {WORKSPACE_BUCKET}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/scores/*_TWAS_SMR_FDR_PTS_feature_list.txt {WORKSPACE_BUCKET}')

## Test associations with PD

### Create an output directory

In [ ]:
!mkdir /home/jupyter/multiTRS/results

### Start with PPMI

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)
library(pROC)

full_results <- data.table()

# Read in the clincal date
PD_pheno_covars <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")

# Read in the surrogate variables
SVs <- fread("/home/jupyter/multiTRS/RNA/clean/PPMI_SVs.txt")

# Join
PD_pheno_covars_SVs <- inner_join(PD_pheno_covars,SVs)

# Read in the PTS scoresfiles and join
scores <- fread("/home/jupyter/multiTRS/scores/PPMI_TWAS_SMR_FDR_scores.txt")
print(ncol(scores))
PD_pheno_covars_SVs_scores <- inner_join(PD_pheno_covars_SVs,scores)
print(nrow(PD_pheno_covars_SVs_scores))

# Define the score names for the regression
score_names <- colnames(scores[,-1])

# TEST:
# score_names <- score_names[1]

# Run the null (covariate only) model
null <- glm(case_control_other_at_baseline ~ age_at_baseline + sex + SV1 + SV2 + SV3 + SV4, data = PD_pheno_covars_SVs_scores, family = "binomial")

null_probs <- predict(null, type = "response")

roc_obj_null <- roc(PD_pheno_covars_SVs_scores$case_control_other_at_baseline, null_probs)

null_auc <- auc(roc_obj_null)

null_auc_ci <- ci.auc(roc_obj_null)

null_auc_lower <- null_auc_ci[1]
null_auc_upper <- null_auc_ci[3]

null_coords <- coords(
  roc_obj_null,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

null_sensitivity <- as.numeric(null_coords["sensitivity"])
null_specificity <- as.numeric(null_coords["specificity"])
null_accuracy <- as.numeric(null_coords["accuracy"])

# Loop over the scores for the full PTS regression models
for (i in score_names){

    
formula <- as.formula(paste0("case_control_other_at_baseline ~ scale(", i, ") + age_at_baseline + sex + SV1 + SV2 + SV3 + SV4"))
  

model <- glm(formula = formula, data = PD_pheno_covars_SVs_scores, family = "binomial")
    
model_probs <- predict(model, type = "response")
    
roc_obj_model <- roc(PD_pheno_covars_SVs_scores$case_control_other_at_baseline, model_probs)

model_auc <- auc(roc_obj_model)

model_auc_ci <- ci.auc(roc_obj_model)

model_auc_lower <- model_auc_ci[1]
model_auc_upper <- model_auc_ci[3]
    
model_coords <- coords(
  roc_obj_model,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])


auc_diff <- model_auc - null_auc
    
r2 <- fmsb::NagelkerkeR2(model)$R2 - fmsb::NagelkerkeR2(null)$R2
    
coefs <- summary(model)$coefficients
    
predictor_row <- coefs[2, ]

delong_test <- roc.test(roc_obj_null, roc_obj_model, method = "delong")

    
results <- data.table(predictor = i,
                      AUC_null = null_auc,
                      AUC_lower_null = null_auc_lower,
                      AUC_upper_null = null_auc_upper,
                      AUC_full = model_auc,
                      AUC_lower_full = model_auc_lower,
                      AUC_upper_full = model_auc_upper,
                      AUC_diff = auc_diff,
                      sens_null = null_sensitivity,
                      sens_full = model_sensitivity,
                      sens_diff = model_sensitivity - null_sensitivity,
                      spec_null = null_specificity,
                      spec_full = model_specificity,
                      spec_diff = model_specificity - null_specificity,
                      acc_null = null_accuracy,
                      acc_full = model_accuracy,
                      acc_diff = model_accuracy - null_accuracy,
                      R2 = r2,
                      estimate = predictor_row["Estimate"],
                      std_error = predictor_row["Std. Error"],
                      p_value = predictor_row["Pr(>|z|)"],
                      delong_z = delong_test$statistic,
                      delong_p = delong_test$p.value)

full_results <- rbind(full_results,results)
    

}

full_results <- full_results %>% arrange(p_value)

full_results$OR <- exp(full_results$estimate)
full_results$OR_lower <- exp(full_results$estimate - 1.96*full_results$std_error)
full_results$OR_upper <- exp(full_results$estimate + 1.96*full_results$std_error)

print(nrow(full_results))

# Print results
write.table(full_results,paste0("/home/jupyter/multiTRS/results/PPMI_FUSION_SMR_FDR_results.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

### For PDBP

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)

full_results <- data.table()

# Read in the clincal date
PD_pheno_covars <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")

# Read in the surrogate variables
SVs <- fread("/home/jupyter/multiTRS/RNA/clean/PDBP_SVs.txt")

# Join
PD_pheno_covars_SVs <- inner_join(PD_pheno_covars,SVs)

# Read in the PTS scoresfiles and join
scores <- fread("/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores.txt")
PD_pheno_covars_SVs_scores <- inner_join(PD_pheno_covars_SVs,scores)
print(nrow(PD_pheno_covars_SVs_scores))

# Define the score names for the regression
score_names <- colnames(scores[,-1])

# TEST:
# score_names <- score_names[1]

# Run the null (covariate only) model
null <- glm(case_control_other_at_baseline ~ age_at_baseline + sex + SV1 + SV2 + SV3 + SV4, data = PD_pheno_covars_SVs_scores, family = "binomial")

null_probs <- predict(null, type = "response")

roc_obj_null <- roc(PD_pheno_covars_SVs_scores$case_control_other_at_baseline, null_probs)

null_auc <- auc(roc_obj_null)

null_auc_ci <- ci.auc(roc_obj_null)

null_auc_lower <- null_auc_ci[1]
null_auc_upper <- null_auc_ci[3]

null_coords <- coords(
  roc_obj_null,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

null_sensitivity <- as.numeric(null_coords["sensitivity"])
null_specificity <- as.numeric(null_coords["specificity"])
null_accuracy <- as.numeric(null_coords["accuracy"])

# Loop over the scores for the full PTS regression models
for (i in score_names){

    
formula <- as.formula(paste0("case_control_other_at_baseline ~ scale(", i, ") + age_at_baseline + sex + SV1 + SV2 + SV3 + SV4"))
  

model <- glm(formula = formula, data = PD_pheno_covars_SVs_scores, family = "binomial")
    
model_probs <- predict(model, type = "response")
    
roc_obj_model <- roc(PD_pheno_covars_SVs_scores$case_control_other_at_baseline, model_probs)

model_auc <- auc(roc_obj_model)
    
model_auc_ci <- ci.auc(roc_obj_model)

model_auc_lower <- model_auc_ci[1]
model_auc_upper <- model_auc_ci[3]

model_coords <- coords(
  roc_obj_model,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])


auc_diff <- model_auc - null_auc
    
r2 <- fmsb::NagelkerkeR2(model)$R2 - fmsb::NagelkerkeR2(null)$R2
    
coefs <- summary(model)$coefficients
    
predictor_row <- coefs[2, ]
    
delong_test <- roc.test(roc_obj_null, roc_obj_model, method = "delong")

    
results <- data.table(predictor = i,
                      AUC_null = null_auc,
                      AUC_lower_null = null_auc_lower,
                      AUC_upper_null = null_auc_upper,
                      AUC_full = model_auc,
                      AUC_lower_full = model_auc_lower,
                      AUC_upper_full = model_auc_upper,
                      AUC_diff = auc_diff,
                      sens_null = null_sensitivity,
                      sens_full = model_sensitivity,
                      sens_diff = model_sensitivity - null_sensitivity,
                      spec_null = null_specificity,
                      spec_full = model_specificity,
                      spec_diff = model_specificity - null_specificity,
                      acc_null = null_accuracy,
                      acc_full = model_accuracy,
                      acc_diff = model_accuracy - null_accuracy,
                      R2 = r2,
                      estimate = predictor_row["Estimate"],
                      std_error = predictor_row["Std. Error"],
                      p_value = predictor_row["Pr(>|z|)"],
                      delong_z = delong_test$statistic,
                      delong_p = delong_test$p.value)

full_results <- rbind(full_results,results)
    

}

full_results <- full_results %>% arrange(p_value)

full_results$OR <- exp(full_results$estimate)
full_results$OR_lower <- exp(full_results$estimate - 1.96*full_results$std_error)
full_results$OR_upper <- exp(full_results$estimate + 1.96*full_results$std_error)

print(nrow(full_results))

# Print results
write.table(full_results,paste0("/home/jupyter/multiTRS/results/PDBP_FUSION_SMR_FDR_results.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

### For HBS

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)
library(ggplot2)
library(pROC)

full_results <- data.table()

# Read in the clincal date
PD_pheno_covars <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")

# Read in the surrogate variables
SVs <- fread("/home/jupyter/multiTRS/RNA/clean/HBS_SVs.txt")

# Join
PD_pheno_covars_SVs <- inner_join(PD_pheno_covars,SVs)

# Read in the PTS scoresfiles and join
scores <- fread("/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores.txt")
PD_pheno_covars_scores <- inner_join(PD_pheno_covars_SVs,scores)
print(nrow(PD_pheno_covars_scores))

# Define the score names for the regression
score_names <- colnames(scores[,-1])

# TEST:
# score_names <- score_names[1]

# Run the null (covariate only) model
null <- glm(case_control_other_at_baseline ~ age_at_baseline + sex, data = PD_pheno_covars_scores, family = "binomial")

null_probs <- predict(null, type = "response")

roc_obj_null <- roc(PD_pheno_covars_scores$case_control_other_at_baseline, null_probs)

null_auc <- auc(roc_obj_null)

null_auc_ci <- ci.auc(roc_obj_null)

null_auc_lower <- null_auc_ci[1]
null_auc_upper <- null_auc_ci[3]

null_coords <- coords(
  roc_obj_null,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

null_sensitivity <- as.numeric(null_coords["sensitivity"])
null_specificity <- as.numeric(null_coords["specificity"])
null_accuracy <- as.numeric(null_coords["accuracy"])


# Loop over the scores for the full PTS regression models
for (i in score_names){

    
formula <- as.formula(paste0("case_control_other_at_baseline ~ scale(", i, ") + age_at_baseline + sex"))
  

model <- glm(formula = formula, data = PD_pheno_covars_scores, family = "binomial")
    
model_probs <- predict(model, type = "response")
    
roc_obj_model <- roc(PD_pheno_covars_scores$case_control_other_at_baseline, model_probs)

model_auc <- auc(roc_obj_model)

model_auc_ci <- ci.auc(roc_obj_model)

model_auc_lower <- model_auc_ci[1]
model_auc_upper <- model_auc_ci[3]

model_coords <- coords(
  roc_obj_model,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])


auc_diff <- model_auc - null_auc
    
r2 <- fmsb::NagelkerkeR2(model)$R2 - fmsb::NagelkerkeR2(null)$R2
    
coefs <- summary(model)$coefficients
    
predictor_row <- coefs[2, ]
    
delong_test <- roc.test(roc_obj_null, roc_obj_model, method = "delong")

    
results <- data.table(predictor = i,
                      AUC_null = null_auc,
                      AUC_lower_null = null_auc_lower,
                      AUC_upper_null = null_auc_upper,
                      AUC_full = model_auc,
                      AUC_lower_full = model_auc_lower,
                      AUC_upper_full = model_auc_upper,
                      AUC_diff = auc_diff,
                      sens_null = null_sensitivity,
                      sens_full = model_sensitivity,
                      sens_diff = model_sensitivity - null_sensitivity,
                      spec_null = null_specificity,
                      spec_full = model_specificity,
                      spec_diff = model_specificity - null_specificity,
                      acc_null = null_accuracy,
                      acc_full = model_accuracy,
                      acc_diff = model_accuracy - null_accuracy,
                      R2 = r2,
                      estimate = predictor_row["Estimate"],
                      std_error = predictor_row["Std. Error"],
                      p_value = predictor_row["Pr(>|z|)"],
                      delong_z = delong_test$statistic,
                      delong_p = delong_test$p.value)

full_results <- rbind(full_results,results)
    

}

full_results <- full_results %>% arrange(p_value)


full_results$OR <- exp(full_results$estimate)
full_results$OR_lower <- exp(full_results$estimate - 1.96*full_results$std_error)
full_results$OR_upper <- exp(full_results$estimate + 1.96*full_results$std_error)

print(nrow(full_results))

# Print results
write.table(full_results,paste0("/home/jupyter/multiTRS/results/HBS_FUSION_SMR_FDR_results.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

### Check files exist and copy over the files to the workspace to save

In [ ]:
!ls /home/jupyter/multiTRS/results/
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/results/*SMR_FDR_results.txt {WORKSPACE_BUCKET}')

## Meta-analyses results across samples with each TRS weighting method

### Fixed effects

In [ ]:
%%R

library(data.table)
library(dplyr)
library(metafor)
library(stringr)

setwd("/home/jupyter/multiTRS/results/")

## First we need to bind the results from the three samples
results_files <- list.files(pattern = "_FUSION_SMR_FDR_results.txt")

print(results_files)

full_results <- data.frame()

for (file in results_files){
    
    results_table <- fread(file)
    cohort <- str_remove_all(file,"_FUSION_SMR_FDR_results.txt")
    results_table$cohort <- cohort
    results_table <- results_table %>% select(cohort, everything())
    
    full_results <- rbind(full_results,results_table)
      
}

write.table(full_results,paste0("/home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_per_cohort_combined.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)


# Obtain unique predictors
predictors <- unique(full_results$predictor)

meta_results <- data.table(
  predictor = character(),
  meta_beta = numeric(),
  meta_se   = numeric(),
  z         = numeric(),
  p_value         = numeric(),
  beta_lower     = numeric(),
  beta_upper     = numeric(),
  Q         = numeric(),
  Q_p_value    = numeric(),
  I2        = numeric(),
  Direction_consistent = character(),
  Majority_sig = character()
)

# Loop over predictors
for (pred in predictors) {
  
  # Filter rows for this predictor
  meta_table <- full_results %>%
    filter(predictor == pred)
    
# Check there is a consistent direction of effect
  Direction_consistent <- if (length(unique(sign(meta_table$estimate[meta_table$estimate != 0]))) == 1) {
  "PASS"
} else {
  "FAIL"
}
    
# Check 2/3 are significant at Bonferroni level   
  Majority_sig <- if (sum(meta_table$p_value <= 0.00009) >= 2) {
  "PASS"
} else {
  "FAIL"
}


  
  # Skip if there are less than 2 studies (can't meta-analyse)
  if(nrow(meta_table) < 2){
      print(paste0("There are less than 2 studies for ", pred, ". Need to investigate..."))
      next 
  }
  
  # Fixed-effect IVW meta-analysis
  res <- rma(yi = estimate,
             sei = std_error,
             data = meta_table,
             method = "FE")  
    
    

  # Extract heterogeneity statistics
  Q_val     <- res$QE
  Q_p       <- res$QEp
  I2_val    <- res$I2
  
  meta_results <- rbind(meta_results, data.table(
    predictor = pred,
    meta_beta = as.numeric(res$beta),
    meta_se   = res$se,
    z         = res$zval,
    p_value   = res$pval,
    beta_lower = res$ci.lb,
    beta_upper = res$ci.ub,
    Q         = Q_val,
    Q_p_value    = Q_p,
    I2        = I2_val,
    Direction_consistent = Direction_consistent,
    Majority_sig = Majority_sig
  ))
}

  meta_results$OR <- exp(meta_results$meta_beta)
  meta_results$OR_lower <- exp(meta_results$beta_lower)
  meta_results$OR_upper <- exp(meta_results$beta_upper)

meta_results <- meta_results %>% arrange(predictor)

meta_results <- meta_results %>%
  mutate(weighting_method = case_when(
    grepl("_COLOC_FDR_fusion", predictor) ~ "FUSION-COLOC",
    grepl("_FDR_fusion", predictor) ~ "FUSION",
    grepl("HEIDI_FDR_SMR_single_SNP", predictor) ~ "SMR-HEIDI",
    grepl("single_SNP", predictor) ~ "SMR",
    grepl("_HEIDI_FDR_SMR", predictor) ~ "SMR-multi-HEIDI",
    grepl("_FDR_SMR", predictor) ~ "SMR-multi",
    TRUE ~ "Other"
  ))



meta_results <- meta_results %>% relocate(weighting_method, .before = meta_beta)

# Perform multiiple testing correction
p_for_correction <- meta_results$p_value

bonferroni <- p.adjust(p_for_correction, method = "bonferroni")

meta_results$p_bonferroni <- bonferroni

meta_results$threshold_pass <- ifelse(
  meta_results$p_bonferroni <= 0.05 &
  meta_results$Direction_consistent == "PASS" &
  meta_results$Majority_sig == "PASS",
  "PASS", "FAIL"
)

meta_results <- meta_results %>% arrange(p_bonferroni)

print(meta_results %>% filter(threshold_pass == "PASS")) 

meta_results_sig <- meta_results %>% filter(threshold_pass == "PASS")

meta_results_sig <- meta_results_sig %>% arrange(predictor)

print(meta_results_sig)

meta_results_sig_list <- meta_results_sig$predictor

print(head(meta_results_sig_list))

write.table(meta_results_sig_list,paste0("/home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_meta_analysed_sig_list.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

meta_results$predictor <- str_remove_all(meta_results$predictor,"_COLOC_FDR_fusion")
meta_results$predictor <- str_remove_all(meta_results$predictor,"_FDR_fusion")
meta_results$predictor <- str_remove_all(meta_results$predictor,"_HEIDI_FDR_SMR")
meta_results$predictor <- str_remove_all(meta_results$predictor,"_FDR_SMR")
meta_results$predictor <- str_remove_all(meta_results$predictor,"_single_SNP")

print(table(meta_results_sig$weighting_method))

print(nrow(meta_results))
    
write.table(meta_results,paste0("/home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_meta_analysed.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

### Random-effects

We can run a random effects meta-analysis on the significant results as a sensitivity analysis

In [ ]:
%%R

library(data.table)
library(dplyr)
library(metafor)
library(stringr)

setwd("/home/jupyter/multiTRS/results/")

sig_list <- fread("/home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_meta_analysed_sig_list.txt")


## First we need to bind the results from the three samples
results_files <- list.files(pattern = "_FUSION_SMR_FDR_results.txt")

print(results_files)

full_results <- data.frame()

for (file in results_files){
    
    results_table <- fread(file)
    cohort <- str_remove_all(file,"_FUSION_SMR_FDR_results.txt")
    results_table$cohort <- cohort
    results_table <- results_table %>% select(cohort, everything())
    
    full_results <- rbind(full_results,results_table)
      
}

full_results <- full_results %>% filter(predictor %in% sig_list$x)

# Obtain unique predictors
predictors <- unique(full_results$predictor)


#predictors <- predictors[grepl("ALS", predictors)]

meta_results <- data.table(
  predictor = character(),
  meta_beta = numeric(),
  meta_se   = numeric(),
  z         = numeric(),
  p_value         = numeric(),
  beta_lower     = numeric(),
  beta_upper     = numeric(),
  tau2 = numeric(),
  Q         = numeric(),
  Q_p_value    = numeric(),
  I2        = numeric()
)


for (pred in predictors) {
  
  # Filter rows for this predictor
  meta_table <- full_results %>%
    filter(predictor == pred)
    


  
  # Skip if there are less than 2 studies
  if(nrow(meta_table) < 2){
      print(paste0("There are less than 2 studies for ", pred, ". Need to investigate..."))
      next 
  }
  
  # REML IVW meta-analysis
  res <- rma(yi = estimate,     # make sure your column names match
             sei = std_error,
             data = meta_table,
             method = "REML")  
    
    

  
  # Extract heterogeneity statistics
  Q_val     <- res$QE
  Q_p       <- res$QEp
  I2_val    <- res$I2
  tau2_val  <- res$tau2
  
  meta_results <- rbind(meta_results, data.table(
    predictor = pred,
    meta_beta = as.numeric(res$beta),
    meta_se   = res$se,
    z         = res$zval,
    p_value   = res$pval,
    beta_lower = res$ci.lb,
    beta_upper = res$ci.ub,
    tau2       = tau2_val,
    Q         = Q_val,
    Q_p_value    = Q_p,
    I2        = I2_val
  ))
}

  meta_results$OR <- exp(meta_results$meta_beta)
  meta_results$OR_lower <- exp(meta_results$beta_lower)
  meta_results$OR_upper <- exp(meta_results$beta_upper)

meta_results <- meta_results %>% arrange(predictor)

meta_results <- meta_results %>%
  mutate(weighting_method = case_when(
    grepl("_COLOC_FDR_fusion", predictor) ~ "FUSION-COLOC",
    grepl("_FDR_fusion", predictor) ~ "FUSION",
    grepl("HEIDI_FDR_SMR_single_SNP", predictor) ~ "SMR-HEIDI",
    grepl("single_SNP", predictor) ~ "SMR",
    grepl("_HEIDI_FDR_SMR", predictor) ~ "SMR-multi-HEIDI",
    grepl("_FDR_SMR", predictor) ~ "SMR-multi",
    TRUE ~ "Other"
  ))



meta_results$predictor <- str_remove_all(meta_results$predictor,"_COLOC_FDR_fusion")
meta_results$predictor <- str_remove_all(meta_results$predictor,"_FDR_fusion")
meta_results$predictor <- str_remove_all(meta_results$predictor,"_HEIDI_FDR_SMR")
meta_results$predictor <- str_remove_all(meta_results$predictor,"_FDR_SMR")
meta_results$predictor <- str_remove_all(meta_results$predictor,"_single_SNP")

meta_results <- meta_results %>% relocate(weighting_method, .before = meta_beta)

meta_results <- meta_results %>% arrange(p_value)

print(meta_results)

print(table(meta_results_sig$weighting_method))

print(nrow(meta_results))
    
write.table(meta_results,paste0("/home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_REML_random_effects_meta_analysed.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

## Liability scale R2

To get an accurate estimate of R2 for a binary outcome iit needs to be on the liability scale. It is therefore required that we obtain an estimate for variance explained on the observed scale using linear regression, as per Lee et al. "A Better Coefﬁcient of Determination for Genetic Proﬁle Analysis", then convert the estimates.

https://pubmed.ncbi.nlm.nih.gov/22714935/

### Run linear regression for each cohort

#### PPMI

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)
library(pROC)

full_results <- data.table()

PD_pheno_covars <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")

SVs <- fread("/home/jupyter/multiTRS/RNA/clean/PPMI_SVs.txt")

PD_pheno_covars_SVs <- inner_join(PD_pheno_covars,SVs, by = "participant_id")

scores <- fread("/home/jupyter/multiTRS/scores/PPMI_TWAS_SMR_FDR_scores.txt")

print(ncol(scores))

PD_pheno_covars_SVs_scores <- inner_join(PD_pheno_covars_SVs,scores,by = "participant_id")

print(nrow(PD_pheno_covars_SVs_scores))

score_names <- colnames(scores[,-1])


null <- lm(case_control_other_at_baseline ~ age_at_baseline + sex + SV1 + SV2 + SV3 + SV4, data = PD_pheno_covars_SVs_scores)


for (i in score_names){

    
formula <- as.formula(paste0("case_control_other_at_baseline ~ scale(", i, ") + age_at_baseline + sex + SV1 + SV2 + SV3 + SV4"))
  

model <- lm(formula = formula, data = PD_pheno_covars_SVs_scores)

    
r2 <- summary(model)$r.squared - summary(null)$r.squared


    
results <- data.table(predictor = i,
                      R2 = r2)

full_results <- rbind(full_results,results)
    

}



print(full_results)


write.table(full_results,paste0("/home/jupyter/multiTRS/results/PPMI_FUSION_SMR_FDR_results_linear_r2.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

#### PDBP

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)
library(pROC)

full_results <- data.table()

PD_pheno_covars <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")

SVs <- fread("/home/jupyter/multiTRS/RNA/clean/PDBP_SVs.txt")

PD_pheno_covars_SVs <- inner_join(PD_pheno_covars,SVs, by = "participant_id")

scores <- fread("/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores.txt")

print(ncol(scores))

PD_pheno_covars_SVs_scores <- inner_join(PD_pheno_covars_SVs,scores,by = "participant_id")

print(nrow(PD_pheno_covars_SVs_scores))

score_names <- colnames(scores[,-1])


null <- lm(case_control_other_at_baseline ~ age_at_baseline + sex + SV1 + SV2 + SV3 + SV4, data = PD_pheno_covars_SVs_scores)


for (i in score_names){

    
formula <- as.formula(paste0("case_control_other_at_baseline ~ scale(", i, ") + age_at_baseline + sex + SV1 + SV2 + SV3 + SV4"))
  

model <- lm(formula = formula, data = PD_pheno_covars_SVs_scores)

    
r2 <- summary(model)$r.squared - summary(null)$r.squared


    
results <- data.table(predictor = i,
                      R2 = r2)

full_results <- rbind(full_results,results)
    

}



print(full_results)


write.table(full_results,paste0("/home/jupyter/multiTRS/results/PDBP_FUSION_SMR_FDR_results_linear_r2.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

#### HBS

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)
library(pROC)

full_results <- data.table()

PD_pheno_covars <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")

SVs <- fread("/home/jupyter/multiTRS/RNA/clean/HBS_SVs.txt")

PD_pheno_covars_SVs <- inner_join(PD_pheno_covars,SVs, by = "participant_id")

scores <- fread("/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores.txt")

print(ncol(scores))

PD_pheno_covars_SVs_scores <- inner_join(PD_pheno_covars_SVs,scores,by = "participant_id")

print(nrow(PD_pheno_covars_SVs_scores))

score_names <- colnames(scores[,-1])


null <- lm(case_control_other_at_baseline ~ age_at_baseline + sex, data = PD_pheno_covars_SVs_scores)


for (i in score_names){

    
formula <- as.formula(paste0("case_control_other_at_baseline ~ scale(", i, ") + age_at_baseline + sex"))
  

model <- lm(formula = formula, data = PD_pheno_covars_SVs_scores)

    
r2 <- summary(model)$r.squared - summary(null)$r.squared


    
results <- data.table(predictor = i,
                      R2 = r2)

full_results <- rbind(full_results,results)
    

}



print(full_results)


write.table(full_results,paste0("/home/jupyter/multiTRS/results/HBS_FUSION_SMR_FDR_results_linear_r2.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

### Meta-analyses R2 values

#### Convert to liability scale

In [ ]:
%%R

library(data.table)
library(dplyr)
library(metafor)
library(stringr)

setwd("/home/jupyter/multiTRS/results/")

## First we need to bind the results from the three samples
results_files <- list.files(pattern = "_FUSION_SMR_FDR_results_linear_r2.txt")

print(results_files)

full_results <- data.frame()

for (file in results_files){
    
    results_table <- fread(file)
    cohort <- str_remove_all(file,"_FUSION_SMR_FDR_results_linear_r2.txt")
    results_table$cohort <- cohort
    results_table <- results_table %>% select(cohort, everything())
    
        if (cohort == "PPMI") {
    results_table$sampprev <- 0.624
} else if (cohort == "PDBP") {
    results_table$sampprev <- 0.628
} else {
    results_table$sampprev <- 0.533
}
    
            if (cohort == "PPMI") {
    results_table$N <- 974
} else if (cohort == "PDBP") {
    results_table$N <- 1102
} else {
    results_table$N <- 655
}
    
    full_results <- rbind(full_results,results_table)

    
      
}

full_results$N_predictors <- 1


# Liability scale for R2 conversion
R2liab <- function(k, r2, p) {
  # K baseline disease risk
  # r2 from a linear regression model attributable to genomic profile risk score
  # P proportion of sample that are cases
  # calculates proportion of variance explained on the liability scale
  #Lee SH, Goddard ME, Wray NR, Visscher PM. (2012) A better coefficient of determination for genetic profile analysis. Genet Epidemiol. 2012 Apr;36(3):214-24.
  x= qnorm(1-k)
  z= dnorm(x)
  i=z/k
  C= k*(1-k)*k*(1-k)/(z^2*p*(1-p))
  theta= i*((p-k)/(1-k))*(i*((p-k)/(1-k))-x)
  R2l = C*r2 / (1 + C*theta*r2)
}

full_results$R2liability_0.005 <- R2liab(k=0.005, r2=full_results$R2, p=full_results$sampprev)

print(full_results)

write.table(full_results,paste0("/home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_liability_scale_r2.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

#### Perform meta-analysis

In [ ]:
%%R

library(data.table)
library(dplyr)
library(metafor)
library(stringr)

all_liability_r2 <- fread("/home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_liability_scale_r2.txt")

# Obtain unique predictors
predictors <- unique(all_liability_r2$predictor)

# Initialize results list
results_list <- vector("list", length(predictors))

for (i in seq_along(predictors)) {
  pred <- predictors[i]
  
  # Filter rows for this predictor
  meta_table <- all_liability_r2 %>%
    filter(predictor == pred)
  
  # Skip if there are less than 2 studies
  if(nrow(meta_table) < 2){
    print(paste0("There are less than 2 studies for ", pred, ". Need to investigate..."))
    next 
  }
  
  # Compute effect sizes for this predictor
  es <- escalc(
    measure = "ZR2", 
    r2i = meta_table$R2liability_0.005, 
    ni  = meta_table$N, 
    mi  = meta_table$N_predictors
  )
  
  # Meta-analysis using fixed effects
  res <- rma(yi = yi, vi = vi, data = es, method = "FE")
  
  # Back-transform using transf.ztor2
  pred_res <- predict(res, transf = transf.ztor2)
  
  results_list[[i]] <- data.table(
    predictor           = pred,
    meta_estimate       = pred_res$pred,
    meta_estimate_lower = pred_res$ci.lb,
    meta_estimate_upper = pred_res$ci.ub,
    z                   = res$zval,
    p_value             = res$pval
  )
}

full_meta_results <- rbindlist(results_list) %>%
  mutate(weighting_method = case_when(
    grepl("_COLOC_FDR_fusion", predictor) ~ "FUSION-COLOC",
    grepl("_FDR_fusion", predictor) ~ "FUSION",
    grepl("HEIDI_FDR_SMR_single_SNP", predictor) ~ "SMR-HEIDI",
    grepl("single_SNP", predictor) ~ "SMR",
    grepl("_HEIDI_FDR_SMR", predictor) ~ "SMR-multi-HEIDI",
    grepl("_FDR_SMR", predictor) ~ "SMR-multi",
    TRUE ~ "Other"
  )) %>%
  mutate(predictor = str_remove_all(predictor, "_COLOC_FDR_fusion"),
         predictor = str_remove_all(predictor, "_FDR_fusion"),
         predictor = str_remove_all(predictor, "_HEIDI_FDR_SMR"),
         predictor = str_remove_all(predictor, "_FDR_SMR"),
         predictor = str_remove_all(predictor, "_single_SNP")) %>%
  arrange(desc(meta_estimate))

print(full_meta_results)

full_meta_results <- full_meta_results %>% select(predictor,weighting_method,meta_estimate) %>% rename(Liability_R2 = meta_estimate)

all_results <- fread("/home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_meta_analysed.txt")

combined <- inner_join(all_results,full_meta_results, by = c("predictor","weighting_method"))

print(combined)

write.table(combined,paste0("/home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_liability_scale_r2_meta_analysed_with_meta_results.txt"), sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

### Transfer across to workspace

In [ ]:
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_liability_scale_r2_meta_analysed_with_meta_results.txt {WORKSPACE_BUCKET}')

In [ ]:
!ls /home/jupyter/multiTRS/results/
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_meta_analysed.txt {WORKSPACE_BUCKET}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_per_cohort_combined.txt {WORKSPACE_BUCKET}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_REML_random_effects_meta_analysed.txt {WORKSPACE_BUCKET}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_liability_scale_r2_meta_analysed.txt {WORKSPACE_BUCKET}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/results/ALL_FUSION_SMR_FDR_results_liability_scale_r2.txt {WORKSPACE_BUCKET}')